# Father, Mother & Son: the motorcycle decision

Each agent sees the conversation from its own point of view: its own lines as its own, everyone else's labelled by name. A moderator then states what was agreed as structured data, and every turn is checked for an agent writing lines for someone else. Runs on any OpenAI-compatible model set in the repo's `.env`.

In [2]:
# Cell 1 — Setup
import json, os, sys
from datetime import date
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
if not os.getenv("OPENAI_BASE_URL") and os.getenv("OPENAI_API_KEY", "lm-studio") in ("", "lm-studio"):
    raise RuntimeError("No LLM endpoint: set OPENAI_BASE_URL (LM Studio) or a real OPENAI_API_KEY in the repo's .env")
PROJECT_DIR = Path.cwd().parent if Path.cwd().name in ("notebooks", "src") else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR / "src"))
import conversation as cv

OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
chat = cv.openai_chat()
MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")
print("Model:", MODEL)

Model: qwen/qwen3.6-35b-a3b


In [3]:
# Cell 2 — The scene and the agents
SCENE = "A family discusses whether the son, 19, may buy a small motorcycle. Keep it respectful, specific and concise."
RUN_NAME = "motorcycle_run"
conv = cv.Conversation(SCENE, [
    cv.Agent("SON", "You want to buy a motorcycle. Give your reasons (independence, budget, safety training), answer questions, and accept reasonable conditions."),
    cv.Agent("MOTHER", "You care about safety and are against the purchase for now. Raise concerns calmly (injury risk, insurance) and suggest alternatives."),
    cv.Agent("FATHER", "You support the idea but want rules: a licence, a safety course, full gear and a clear budget. Help the family reach a compromise."),
])
OPENING = ("SON", "I'd like to buy a small motorcycle. I've looked into safety courses and budgeted for gear, and I think I can handle it responsibly.")
ORDER, ROUNDS = ["MOTHER", "FATHER", "SON"], 4

In [4]:
# Cell 3 — Run the conversation
log = cv.run(conv, chat, OPENING, ORDER, rounds=ROUNDS)
for t in log:
    flag = f"   [speaks for {', '.join(t['speaks_for_others'])}]" if t["speaks_for_others"] else ""
    print(f"{t['speaker']}: {t['text']}{flag}\n")

SON: I'd like to buy a small motorcycle. I've looked into safety courses and budgeted for gear, and I think I can handle it responsibly.

MOTHER: I appreciate how thoughtfully you've researched safety courses and budgeted for gear, but I remain concerned about the inherent injury risk while you're still building your riding experience. We'd also need to carefully review long-term insurance costs and liability coverage before moving forward, and I'd prefer we wait until you've completed a certified advanced rider course. In the meantime, could we look into getting you a reliable bicycle or a low-speed electric scooter to help you develop road confidence first?

FATHER: I appreciate your caution, Mom, and we can address your concerns by requiring a certified beginner course and a detailed insurance review before any purchase. I fully support him moving forward with the motorcycle, provided he secures his license, commits to a full set of certified gear, and adheres to the clear budget he

In [5]:
# Cell 4 — What did they agree, and did every agent stay in character?
summary = cv.summarize(conv, cv.openai_chat(temperature=0, max_tokens=400))
leaks = [t for t in log if t["speaks_for_others"]]
print(json.dumps(summary, indent=2))
print(f"\nTurns: {len(log)} | turns where an agent spoke for someone else: {len(leaks)}")

json.dump({"model": MODEL, "date": date.today().isoformat(), "scene": SCENE, "transcript": log,
           "summary": summary, "turns_speaking_for_others": len(leaks)},
          open(OUTPUT_DIR / f"{RUN_NAME}.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
print(f"Saved outputs/{RUN_NAME}.json")

{
  "agreed": true,
  "terms": [
    "Son will immediately enroll in a certified beginner safety course",
    "Son will use a bicycle or electric scooter for daily commuting during the training period",
    "Son will adhere to the strict budget he outlined",
    "Son will earn his motorcycle license and maintain an accident-free record for six months",
    "Son will gather formal long-term insurance quotes",
    "The family will meet on a firm review date to reassess the purchase",
    "The milestones will be documented in a written agreement"
  ],
  "open_issues": [
    "Final approval of the motorcycle purchase is pending the review date",
    "Specific long-term insurance costs for a new rider remain to be evaluated",
    "Mother's concerns about injury risk require more riding experience to alleviate",
    "The exact date for the final review meeting has not been set"
  ]
}

Turns: 13 | turns where an agent spoke for someone else: 0
Saved outputs/motorcycle_run.json
